# Word Importance Explorer

**Assignment:** Use TF-IDF on 5 documents and identify top keywords with explanation.

We will build a tiny TF-IDF pipeline from scratch (no external libraries) so the notebook runs anywhere.


## Documents
We use five short documents on different topics to make keyword differences obvious.


In [ ]:
# Five sample documents

docs = [
    "Solar energy powers homes with clean electricity from sunlight.",
    "Electric cars reduce pollution and improve city air quality.",
    "Healthy diets include vegetables, fruits, and regular exercise.",
    "Machine learning models learn patterns from data and make predictions.",
    "Wildlife conservation protects habitats and endangered species."
]

for i, d in enumerate(docs, 1):
    print(f"Doc {i}: {d}")


## TF-IDF (Term Frequency ? Inverse Document Frequency)

We measure how important a word is to a document *relative to the whole set*.

- **TF** (term frequency): how often the word appears in the document.
- **IDF** (inverse document frequency): down-weights words that appear in many documents.

We use:

`tfidf(term, doc) = tf(term, doc) * ( log((N + 1) / (df + 1)) + 1 )`

Where:
- `N` = number of documents (5)
- `df` = number of documents containing the term


In [ ]:
import math
import re
from collections import Counter, defaultdict

stopwords = {
    "the", "and", "a", "an", "with", "from", "to", "of", "in", "on", "for", "is", "are",
    "that", "this", "as", "be", "by", "we", "use", "it", "into"
}


def tokenize(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    return [t for t in tokens if t not in stopwords]

# Tokenize docs

doc_tokens = [tokenize(d) for d in docs]

# Document frequency (df)

df = Counter()
for tokens in doc_tokens:
    for term in set(tokens):
        df[term] += 1

N = len(docs)

# IDF per term
idf = {term: math.log((N + 1) / (df_val + 1)) + 1 for term, df_val in df.items()}


In [ ]:
# Compute TF-IDF per document

def tfidf_for_doc(tokens):
    counts = Counter(tokens)
    total = sum(counts.values())
    tfidf = {}
    for term, c in counts.items():
        tf = c / total
        tfidf[term] = tf * idf[term]
    return tfidf

# Top keywords per document

top_k = 5
for i, tokens in enumerate(doc_tokens, 1):
    scores = tfidf_for_doc(tokens)
    top_terms = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    print(f"Doc {i} top terms:")
    for term, score in top_terms:
        print(f"  {term:<15} {score:.3f}")
    print()


## Interpretation

The top terms for each document are the most **distinctive** words for that document.
Common words across the set receive lower scores because their IDF is smaller.
This makes TF-IDF useful for keyword extraction and document summarization.


In [ ]:
# Global view: terms with highest average TF-IDF across all docs

all_scores = defaultdict(list)
for tokens in doc_tokens:
    scores = tfidf_for_doc(tokens)
    for term, score in scores.items():
        all_scores[term].append(score)

avg_scores = {term: sum(vals)/len(vals) for term, vals in all_scores.items()}

print("Top terms overall (average TF-IDF):")
for term, score in sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {term:<15} {score:.3f}")


## Conclusion

We computed TF-IDF on 5 documents and extracted the top keywords per document.
These keywords are the most informative terms for each document because they are frequent **within** a document but uncommon **across** the full set.
